In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

fatal: destination path 'CTAB-GAN-Plus' already exists and is not an empty directory.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer,
)

from sdv.evaluation.single_table import evaluate_quality

from model.ctabgan import CTABGAN

# ----------------------------------------------------
# Load Dataset (Alzheimer's Disease)
# Prefer raw Excel with Subject ID (needed for participant splits).
# Skip cleaned CSVs that already dropped Subject ID.
# ----------------------------------------------------
_here = Path.cwd().resolve()
candidate_paths = []
for _root in [_here, *_here.parents]:
    candidate_paths.extend(
        [
            _root / "Datasets" / "Alzhimers.xlsx",
            _root / "Datasets" / "Alzheimer.xlsx",
            _root / "Alzhimers.xlsx",
            _root / "Alzheimer.xlsx",
        ]
    )
# Local copies last (often pre-cleaned without Subject ID)
candidate_paths.extend(
    [
        _here / "Alzhimers.xlsx",
        _here / "Alzheimer.xlsx",
        _here / "clean_alzheimer.csv",
        _here / ".." / "Other GANS" / "clean_alzheimer.csv",
        _here / ".." / ".." / "Other GANS" / "clean_alzheimer.csv",
        _here / ".." / ".." / ".." / "Datasets" / "Alzhimers.xlsx",
    ]
)

def _load_alzheimer_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    return pd.read_csv(path)

data_path = None
raw_data = None
for p in candidate_paths:
    p = Path(p)
    if not p.exists():
        continue
    df = _load_alzheimer_table(p)
    # Require Subject ID for participant-level evaluation
    if "Subject ID" not in df.columns:
        print(f"Skipping {p} (no 'Subject ID' column)")
        continue
    data_path = p
    raw_data = df
    break

if raw_data is None:
    raise FileNotFoundError(
        "Alzheimer dataset with 'Subject ID' not found. "
        "Expected Datasets/Alzhimers.xlsx under the repo root."
    )

print(f"Loading: {data_path}")

target_col = "Group"
subject_col = "Subject ID"

# Keep Subject ID for participant-level splits; drop other IDs / unused cols
ad_data = raw_data.drop(columns=["M/F", "MRI ID", "Hand"], errors="ignore")
if subject_col not in ad_data.columns:
    raise KeyError(f"{subject_col} is required for participant-level train/test splits")

ad_data[target_col] = ad_data[target_col].replace(
    {"Demented": 1, "Nondemented": 0, "Converted": 1}
)
ad_data[target_col] = pd.to_numeric(ad_data[target_col], errors="coerce")

# Complete-case only — no mean / median / mode imputation
_before = len(ad_data)
_n_missing_rows = int(ad_data.isna().any(axis=1).sum())
ad_data = ad_data.dropna().reset_index(drop=True)
ad_data[target_col] = ad_data[target_col].astype(int)
print(
    f"Dropped {_before - len(ad_data)} rows with missing feature values "
    f"(complete-case; no imputation; rows_with_na={_n_missing_rows})"
)
assert not ad_data.isna().any().any(), "Unexpected NaNs remain after dropna"

def _participant_split(df, subject_col=subject_col, label_col=target_col, test_size=0.2, seed=42):
    """Split by Subject ID so all sessions of a participant stay in one fold."""
    subj_label = df.groupby(subject_col)[label_col].first()
    subjects = subj_label.index.to_numpy()
    subj_y = subj_label.to_numpy()
    subj_train, subj_test = train_test_split(
        subjects, test_size=test_size, random_state=seed, stratify=subj_y
    )
    assert set(subj_train).isdisjoint(set(subj_test)), "Subject overlap between train and test"
    train_df = (
        df[df[subject_col].isin(subj_train)]
        .drop(columns=[subject_col])
        .reset_index(drop=True)
    )
    test_df = (
        df[df[subject_col].isin(subj_test)]
        .drop(columns=[subject_col])
        .reset_index(drop=True)
    )
    print(
        f"Participant split: subjects={len(subjects)} "
        f"(train={len(subj_train)}, test={len(subj_test)}); "
        f"sessions train={len(train_df)}, test={len(test_df)}"
    )
    print("Train Group:", train_df[label_col].value_counts().to_dict())
    print("Test Group:", test_df[label_col].value_counts().to_dict())
    return train_df, test_df

# Placeholder until SEED/TEST_SIZE are set below; single-run cell re-applies split
alzheimer_data = ad_data.drop(columns=[subject_col]).copy()
X = alzheimer_data.drop(columns=[target_col])
y = alzheimer_data[target_col]
train_real = test_real = None

# ----------------------------------------------------
# Metadata
# ----------------------------------------------------
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(alzheimer_data)

# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------------------------------
# Storage Containers
# ----------------------------------------------------
scores = {}

synthetic_datasets = {}

quality_results = []


Loading: /home/gopi.battineni/SYNTH_BENCHMARK/SYNTH/Datasets/Alzhimers.xlsx
Dropped 19 rows with missing feature values (complete-case; no imputation; rows_with_na=19)


In [3]:
# ---------------------------------------------------
# SINGLE RUN
# ---------------------------------------------------

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# ---------------------------------------------------
# TRAIN / TEST SPLIT (NO LEAKAGE)
# ---------------------------------------------------

# Participant-level split (no subject leakage across train/test)
train_real, test_real = _participant_split(
    ad_data, subject_col=subject_col, label_col=target_col, test_size=TEST_SIZE, seed=seed
)
alzheimer_data = train_real.copy()  # generators fit on train subjects only
X = pd.concat([train_real, test_real], ignore_index=True).drop(columns=[target_col])
y = pd.concat([train_real, test_real], ignore_index=True)[target_col]

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# CTABGAN
# ---------------------------------------------------

try:

    data_path = "alzheimer_train.csv"
    train_real.to_csv(data_path, index=False)

    ctabgan = CTABGAN(
        raw_csv_path=data_path,
        categorical_columns=[],
        log_columns=[],
        mixed_columns={},
        integer_columns=[target_col],
        problem_type={"Classification": target_col}
    )

    ctabgan.fit()

    synthetic_ctabgan = ctabgan.data_prep.inverse_prep(
        ctabgan.synthesizer.sample(N_SAMPLES)
    )

    synthetic_ctabgan[target_col] = (
        pd.to_numeric(synthetic_ctabgan[target_col], errors="coerce")
        .round()
        .clip(0, 1)
        .astype(int)
    )

    synthetic_datasets["CTABGAN"] = synthetic_ctabgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_ctabgan,
        metadata=train_metadata
    )

    score = quality.get_score()

    scores["CTABGAN"] = score

    print("CTABGAN:", round(score, 4))

except Exception as e:
    print("CTABGAN Failed:", e)



================ SINGLE RUN ================
Participant split: subjects=142 (train=113, test=29); sessions train=284, test=70
Train Group: {0: 151, 1: 133}
Test Group: {0: 39, 1: 31}


100%|██████████| 150/150 [00:32<00:00,  4.67it/s]


Finished training in 34.04462695121765  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 246.31it/s]|
Column Shapes Score: 60.23%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 306.20it/s]|
Column Pair Trends Score: 45.0%

Overall Score (Average): 52.62%

CTABGAN: 0.5262


In [4]:
# WGAN-GP

try:

    import traceback

    data_wgan = train_real.copy()

    encoder = LabelEncoder()
    data_wgan[target_col] = encoder.fit_transform(data_wgan[target_col])

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_wgan)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    real_tensor = torch.tensor(
        scaled_data,
        dtype=torch.float32
    )

    batch_size = 64
    latent_dim = 64
    data_dim = real_tensor.shape[1]

    loader = torch.utils.data.DataLoader(
        real_tensor,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    class Generator(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.LayerNorm(128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 256),
                nn.LayerNorm(256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, data_dim)
            )

        def forward(self, z):
            return self.model(z)

    class Critic(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(data_dim, 256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, 128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 1)
            )

        def forward(self, x):
            return self.model(x)

    generator = Generator().to(device)
    critic = Critic().to(device)

    optimizer_G = optim.Adam(
        generator.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    optimizer_C = optim.Adam(
        critic.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    def gradient_penalty(critic, real_samples, fake_samples):

        alpha = torch.rand(real_samples.size(0), 1, device=device)
        alpha = alpha.expand_as(real_samples)

        interpolates = (
            alpha * real_samples +
            (1 - alpha) * fake_samples
        ).requires_grad_(True)

        critic_interpolates = critic(interpolates)

        gradients = torch.autograd.grad(
            outputs=critic_interpolates,
            inputs=interpolates,
            grad_outputs=torch.ones_like(critic_interpolates),
            create_graph=True,
            retain_graph=True
        )[0]

        gradients = gradients.view(gradients.size(0), -1)

        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    for epoch in range(100):

        for real_batch in loader:

            real_batch = real_batch.to(device)

            for _ in range(5):

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake_batch = generator(z).detach()

                critic_real = critic(real_batch).mean()
                critic_fake = critic(fake_batch).mean()

                gp = gradient_penalty(
                    critic,
                    real_batch,
                    fake_batch
                )

                critic_loss = (
                    critic_fake
                    - critic_real
                    + 10 * gp
                )

                optimizer_C.zero_grad()
                critic_loss.backward()
                optimizer_C.step()

            z = torch.randn(
                real_batch.size(0),
                latent_dim,
                device=device
            )

            fake = generator(z)

            generator_loss = -critic(fake).mean()

            optimizer_G.zero_grad()
            generator_loss.backward()
            optimizer_G.step()

    generator.eval()

    with torch.no_grad():

        z = torch.randn(
            N_SAMPLES,
            latent_dim,
            device=device
        )

        synthetic_scaled = generator(z).cpu().numpy()

    synthetic = scaler.inverse_transform(synthetic_scaled)

    synthetic_wgan = pd.DataFrame(
        synthetic,
        columns=data_wgan.columns
    )

    synthetic_wgan[target_col] = (
        synthetic_wgan[target_col]
        .round()
        .clip(0, 1)
        .astype(int)
    )

    synthetic_datasets["WGAN_GP"] = synthetic_wgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_wgan,
        metadata=train_metadata
    )

    scores["WGAN_GP"] = quality.get_score()

    print("WGAN_GP:", round(scores["WGAN_GP"], 4))

    del generator
    del critic
    del real_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("WGAN_GP Failed:")
    traceback.print_exc()

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 227.46it/s]|
Column Shapes Score: 66.97%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 301.25it/s]|
Column Pair Trends Score: 41.2%

Overall Score (Average): 54.08%

WGAN_GP: 0.5408


In [5]:
# SDV MODELS

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        synthetic_data[target_col] = (
            pd.to_numeric(synthetic_data[target_col], errors="coerce")
            .round()
            .clip(0, 1)
            .astype(int)
        )

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 859.06it/s]|
Column Shapes Score: 84.57%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 341.71it/s]|
Column Pair Trends Score: 62.31%

Overall Score (Average): 73.44%

CTGAN: 0.7344
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 713.00it/s]|
Column Shapes Score: 80.37%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 359.86it/s]|
Column Pair Trends Score: 59.75%

Overall Score (Average): 70.06%

CopulaGAN: 0.7006
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 489.99it/s]|
Column Shapes Score: 84.47%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 308.45it/s]|
Column Pair Trends Score: 72.05%

Overall Score (Average): 78.26%

TVAE: 0.7826
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 889.48it/s]|
Column Shapes 

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import pandas as pd

In [8]:
# TRTR (Train Real, Test Real)

print("--- Starting TRTR Evaluation (Train Real, Test Real) ---")

trtr_results = []

SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

for model_name, model in models.items():

    accuracy_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    print(f"Running {model_name}...")

    for seed in SEEDS:

        # Fixed participant fold; seed only changes classifier RNG
        X_train_real = train_real.drop(columns=[target_col])
        y_train_real = train_real[target_col]
        X_test_real = test_real.drop(columns=[target_col])
        y_test_real = test_real[target_col]

        clf = clone(model)

        if hasattr(clf, "random_state"):
            clf.set_params(random_state=seed)

        clf.fit(X_train_real, y_train_real)

        y_pred = clf.predict(X_test_real)

        accuracy_scores.append(
            accuracy_score(y_test_real, y_pred)
        )

        f1_scores.append(
            f1_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        precision_scores.append(
            precision_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        recall_scores.append(
            recall_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

    acc_mean = np.mean(accuracy_scores)
    acc_std = np.std(accuracy_scores)

    f1_mean = np.mean(f1_scores)
    f1_std = np.std(f1_scores)

    prec_mean = np.mean(precision_scores)
    prec_std = np.std(precision_scores)

    rec_mean = np.mean(recall_scores)
    rec_std = np.std(recall_scores)

    trtr_results.append({
        "Model": model_name,
        "Accuracy (Mean±Std)_TRTR": f"{acc_mean:.4f} ± {acc_std:.4f}",
        "F1 (Mean±Std)_TRTR": f"{f1_mean:.4f} ± {f1_std:.4f}",
        "Precision (Mean±Std)_TRTR": f"{prec_mean:.4f} ± {prec_std:.4f}",
        "Recall (Mean±Std)_TRTR": f"{rec_mean:.4f} ± {rec_std:.4f}"
    })

trtr_results_df = pd.DataFrame(trtr_results)

display(
    trtr_results_df[
        [
            "Model",
            "Accuracy (Mean±Std)_TRTR",
            "F1 (Mean±Std)_TRTR",
            "Precision (Mean±Std)_TRTR",
            "Recall (Mean±Std)_TRTR"
        ]
    ]
)


--- Starting TRTR Evaluation (Train Real, Test Real) ---
Running LogReg...
Running SVM-RBF...
Running KNN...
Running NaiveBayes...
Running DecisionTree...
Running RandomForest...
Running ExtraTrees...
Running GradientBoost...
Running AdaBoost...
Running MLP...


,Model,Accuracy (Mean±Std)_TRTR,F1 (Mean±Std)_TRTR,Precision (Mean±Std)_TRTR,Recall (Mean±Std)_TRTR
0,LogReg,0.9857 ± 0.0000,0.9857 ± 0.0000,0.9862 ± 0.0000,0.9857 ± 0.0000
1,SVM-RBF,0.5429 ± 0.0000,0.4349 ± 0.0000,0.4857 ± 0.0000,0.5429 ± 0.0000
2,KNN,0.5429 ± 0.0000,0.5382 ± 0.0000,0.5371 ± 0.0000,0.5429 ± 0.0000
3,NaiveBayes,0.9714 ± 0.0000,0.9715 ± 0.0000,0.9732 ± 0.0000,0.9714 ± 0.0000
4,DecisionTree,0.9186 ± 0.0091,0.9188 ± 0.0091,0.9297 ± 0.0089,0.9186 ± 0.0091
5,RandomForest,0.9700 ± 0.0043,0.9701 ± 0.0043,0.9719 ± 0.0037,0.9700 ± 0.0043
6,ExtraTrees,0.9586 ± 0.0077,0.9587 ± 0.0077,0.9622 ± 0.0064,0.9586 ± 0.0077
7,GradientBoost,0.9714 ± 0.0000,0.9715 ± 0.0000,0.9732 ± 0.0000,0.9714 ± 0.0000
8,AdaBoost,0.9571 ± 0.0000,0.9573 ± 0.0000,0.9609 ± 0.0000,0.9571 ± 0.0000
9,MLP,0.6257 ± 0.1169,0.5594 ± 0.1614,0.6476 ± 0.1688,0.6257 ± 0.1169


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col=None,
    models=None,
    test_size=0.2,
    seeds=[42,43,44,45,46,47,48,49,50,51],
    label=None,
    resplit=False,
):
    """Evaluate classifiers with Acc/F1/Precision/Recall.

    For Alzheimer participant splits, keep resplit=False so sessions stay in the
    fixed train_real / test_real folds (seed only varies classifier RNG).
    """
    if label is not None and label_col is None:
        label_col = label
    if label_col is None:
        raise ValueError("label_col is required")
    if models is None:
        raise ValueError("models is required")

    results = []
    train_df = train_df.copy()
    test_df = test_df.copy()
    # Drop Subject ID if still present
    for _df in (train_df, test_df):
        if "Subject ID" in _df.columns:
            _df.drop(columns=["Subject ID"], inplace=True)
    train_df[label_col] = pd.to_numeric(train_df[label_col], errors="coerce").astype(int)
    test_df[label_col] = pd.to_numeric(test_df[label_col], errors="coerce").astype(int)

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]
            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            if resplit:
                X_train, _, y_train, _ = train_test_split(
                    X_train,
                    y_train,
                    test_size=test_size,
                    random_state=seed,
                    stratify=y_train
                )
                _, X_test, _, y_test = train_test_split(
                    X_test,
                    y_test,
                    test_size=test_size,
                    random_state=seed,
                    stratify=y_test
                )

            # Complete-case residual drop (no imputation)
            train_keep = ~X_train.isna().any(axis=1)
            test_keep = ~X_test.isna().any(axis=1)
            X_train, y_train = X_train.loc[train_keep], y_train.loc[train_keep]
            X_test, y_test = X_test.loc[test_keep], y_test.loc[test_keep]

            scaler = StandardScaler().fit(X_train)

            X_train_s = scaler.transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            clf.fit(X_train_s, y_train)

            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(
                accuracy_score(y_test, y_pred)
            )

            f1_scores.append(
                f1_score(
                    y_test,
                    y_pred,
                    pos_label=1,
                    average="binary",
                    zero_division=0
                )
            )

            precision_scores.append(
                precision_score(
                    y_test,
                    y_pred,
                    pos_label=1,
                    average="binary",
                    zero_division=0
                )
            )

            recall_scores.append(
                recall_score(
                    y_test,
                    y_pred,
                    pos_label=1,
                    average="binary",
                    zero_division=0
                )
            )

        results.append({
            "Model": name,

            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": np.std(accuracy_scores),

            "F1 Mean": np.mean(f1_scores),
            "F1 Std": np.std(f1_scores),

            "Precision Mean": np.mean(precision_scores),
            "Precision Std": np.std(precision_scores),

            "Recall Mean": np.mean(recall_scores),
            "Recall Std": np.std(recall_scores),

            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False
    )


In [10]:
import pandas as pd

label_col = "Group"

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "WGAN_GP",
    "CTABGAN"
]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=train_real,
    test_df=test_real,
    label_col="Group",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds,
    resplit=False,
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=test_real,
        label_col="Group",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds,
        resplit=False,
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
0,LogReg,0.9857 ± 0.0000,0.9841 ± 0.0000,0.9688 ± 0.0000,1.0000 ± 0.0000
1,SVM-RBF,0.9857 ± 0.0000,0.9841 ± 0.0000,0.9688 ± 0.0000,1.0000 ± 0.0000
5,RandomForest,0.9771 ± 0.0070,0.9749 ± 0.0075,0.9511 ± 0.0144,1.0000 ± 0.0000
3,NaiveBayes,0.9714 ± 0.0000,0.9688 ± 0.0000,0.9394 ± 0.0000,1.0000 ± 0.0000
7,GradientBoost,0.9714 ± 0.0000,0.9688 ± 0.0000,0.9394 ± 0.0000,1.0000 ± 0.0000
6,ExtraTrees,0.9586 ± 0.0077,0.9554 ± 0.0079,0.9147 ± 0.0146,1.0000 ± 0.0000
8,AdaBoost,0.9571 ± 0.0000,0.9538 ± 0.0000,0.9118 ± 0.0000,1.0000 ± 0.0000
9,MLP,0.9429 ± 0.0111,0.9380 ± 0.0126,0.9020 ± 0.0116,0.9774 ± 0.0252
4,DecisionTree,0.9329 ± 0.0091,0.9291 ± 0.0096,0.8727 ± 0.0124,0.9935 ± 0.0129
2,KNN,0.9143 ± 0.0000,0.8929 ± 0.0000,1.0000 ± 0.0000,0.8065 ± 0.0000


CTGAN - TSTR


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
3,NaiveBayes,0.7143 ± 0.0000,0.5238 ± 0.0000,1.0000 ± 0.0000,0.3548 ± 0.0000
0,LogReg,0.5571 ± 0.0000,0.0000 ± 0.0000,0.0000 ± 0.0000,0.0000 ± 0.0000
1,SVM-RBF,0.5143 ± 0.0000,0.0556 ± 0.0000,0.2000 ± 0.0000,0.0323 ± 0.0000
8,AdaBoost,0.5143 ± 0.0000,0.0556 ± 0.0000,0.2000 ± 0.0000,0.0323 ± 0.0000
6,ExtraTrees,0.4800 ± 0.0368,0.1534 ± 0.0410,0.2801 ± 0.0758,0.1065 ± 0.0290
5,RandomForest,0.4200 ± 0.0357,0.1578 ± 0.0350,0.2234 ± 0.0519,0.1226 ± 0.0281
7,GradientBoost,0.4143 ± 0.0000,0.0465 ± 0.0000,0.0833 ± 0.0000,0.0323 ± 0.0000
2,KNN,0.4000 ± 0.0000,0.2500 ± 0.0000,0.2800 ± 0.0000,0.2258 ± 0.0000
9,MLP,0.3686 ± 0.0403,0.2596 ± 0.0526,0.2694 ± 0.0487,0.2516 ± 0.0573
4,DecisionTree,0.3643 ± 0.0241,0.2569 ± 0.0315,0.2661 ± 0.0310,0.2484 ± 0.0324


CTGAN - TRTR vs TSTR


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
8,AdaBoost,0.9000 ± 0.0000,0.8772 ± 0.0000,0.9615 ± 0.0000,0.8065 ± 0.0000
5,RandomForest,0.7857 ± 0.0373,0.6986 ± 0.0678,0.9130 ± 0.0290,0.5710 ± 0.0878
7,GradientBoost,0.7771 ± 0.0194,0.7017 ± 0.0304,0.8607 ± 0.0314,0.5935 ± 0.0387
0,LogReg,0.7714 ± 0.0000,0.6522 ± 0.0000,1.0000 ± 0.0000,0.4839 ± 0.0000
6,ExtraTrees,0.7586 ± 0.0259,0.6391 ± 0.0534,0.9370 ± 0.0075,0.4871 ± 0.0585
2,KNN,0.7429 ± 0.0000,0.6250 ± 0.0000,0.8824 ± 0.0000,0.4839 ± 0.0000
1,SVM-RBF,0.7286 ± 0.0000,0.5778 ± 0.0000,0.9286 ± 0.0000,0.4194 ± 0.0000
3,NaiveBayes,0.7286 ± 0.0000,0.6275 ± 0.0000,0.8000 ± 0.0000,0.5161 ± 0.0000
9,MLP,0.7057 ± 0.0462,0.6178 ± 0.0622,0.7294 ± 0.0751,0.5387 ± 0.0646
4,DecisionTree,0.6071 ± 0.0340,0.5121 ± 0.0227,0.5754 ± 0.0557,0.4645 ± 0.0258


CTABGAN - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,CTABGAN,LogReg,0.214286,0.331953,-0.031250,0.516129,0.9857 ± 0.0000,0.7714 ± 0.0000
1,CTABGAN,SVM-RBF,0.257143,0.406349,0.040179,0.580645,0.9857 ± 0.0000,0.7286 ± 0.0000
2,CTABGAN,RandomForest,0.191429,0.276318,0.038099,0.429032,0.9771 ± 0.0070,0.7857 ± 0.0373
3,CTABGAN,NaiveBayes,0.242857,0.341299,0.139394,0.483871,0.9714 ± 0.0000,0.7286 ± 0.0000
4,CTABGAN,GradientBoost,0.194286,0.267016,0.078684,0.406452,0.9714 ± 0.0000,0.7771 ± 0.0194
5,CTABGAN,ExtraTrees,0.200000,0.316248,-0.022356,0.512903,0.9586 ± 0.0077,0.7586 ± 0.0259
6,CTABGAN,AdaBoost,0.057143,0.076653,-0.049774,0.193548,0.9571 ± 0.0000,0.9000 ± 0.0000
7,CTABGAN,MLP,0.237143,0.320187,0.172594,0.438710,0.9429 ± 0.0111,0.7057 ± 0.0462
8,CTABGAN,DecisionTree,0.325714,0.417033,0.297318,0.529032,0.9329 ± 0.0091,0.6071 ± 0.0340
9,CTABGAN,KNN,0.171429,0.267857,0.117647,0.322581,0.9143 ± 0.0000,0.7429 ± 0.0000


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
4,TVAE,0.081571,0.070418,0.133451,-0.017419
3,GaussianCopula,0.087857,0.091061,0.120751,0.056452
0,CTABGAN,0.209143,0.302091,0.078053,0.441290
5,WGAN_GP,0.226714,0.188725,0.307061,-0.003871
1,CTGAN,0.485000,0.779082,0.656616,0.837097
2,CopulaGAN,0.518857,0.691316,0.634183,0.739355


In [11]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
